In [0]:
import pandas as pd
from abc import ABC, abstractmethod

from lib.template.land_to_raw import *


In [0]:
batches = ['20251101_000000', '20251102_000000', '20251103_000000']
table_names = ['raw_cat_cie_10', 'raw_certificates', 'raw_claims', 'raw_consultas', 'raw_terms'] 

domain = 'medallion'
environment = 'dev'

schema_name = 'bronze_healthsys'
catalog_name = domain + '_' + environment

batch_id = batches[2]
for table_name in table_names:

    print('Processing ' + table_name + ' with batch ' + batch_id)
    path = None
    factory_land_to_raw = None

    if table_name == 'raw_cat_cie_10':
        path = f"dbfs:/Workspace/Users/armando.n90@gmail.com/databricks_case/lakehouse/landing/healthsys/{batch_id}/cat_cie_10.csv"
        factory_land_to_raw = FactoryLandToRawFromCsv(path)

    elif table_name == 'raw_certificates':
        path = f"dbfs:/Workspace/Users/armando.n90@gmail.com/databricks_case/lakehouse/landing/healthsys/{batch_id}/certificates.csv"
        factory_land_to_raw = FactoryLandToRawFromCsv(path)

    elif table_name == 'raw_claims':
        path = f"dbfs:/Workspace/Users/armando.n90@gmail.com/databricks_case/lakehouse/landing/healthsys/{batch_id}/claims.csv"
        factory_land_to_raw = FactoryLandToRawFromCsv(path)

    elif table_name == 'raw_consultas':
        path = f"dbfs:/Workspace/Users/armando.n90@gmail.com/databricks_case/lakehouse/landing/healthsys/{batch_id}/consultas.csv"
        factory_land_to_raw = FactoryLandToRawFromCsvWithSemicolon(path)

    elif table_name == 'raw_terms':
        path = f"dbfs:/Workspace/Users/armando.n90@gmail.com/databricks_case/lakehouse/landing/healthsys/{batch_id}/terms.csv"
        factory_land_to_raw = FactoryLandToRawFromCsv(path)

    factory_land_to_raw.create()

    table_processor = LandToRawTemplate(catalog_name=catalog_name, schema_name=schema_name, table_name=table_name, batch_id=batch_id)  
    table_processor.set_component_factory(factory_land_to_raw) 
    table_processor.process()

    print('')
    print(table_processor.metadata)
    print(table_processor.metrics)
    print('')
    table_processor.dataframe.show(3)
    print('')

In [0]:
#table = 'medallion_dev.bronze_healthsys.raw_cat_cie_10'
#table = 'medallion_dev.bronze_healthsys.raw_certificates'
table = 'medallion_dev.bronze_healthsys.raw_claims'
#table = 'medallion_dev.bronze_healthsys.raw_consultas'
#table = 'medallion_dev.bronze_healthsys.raw_terms'
test = spark.read.table(table)
test.printSchema()
test.show(5)

In [0]:

test = spark.read.table('governance_prod.metrics.ingestions')
print(test.count())
test.show(20)


In [0]:
spark.sql('DROP TABLE IF EXISTS medallion_dev.bronze_healthsys.raw_cat_cie_10')
spark.sql('DROP TABLE IF EXISTS medallion_dev.bronze_healthsys.raw_certificates')
spark.sql('DROP TABLE IF EXISTS medallion_dev.bronze_healthsys.raw_claims')
spark.sql('DROP TABLE IF EXISTS medallion_dev.bronze_healthsys.raw_consultas')
spark.sql('DROP TABLE IF EXISTS medallion_dev.bronze_healthsys.raw_terms')
spark.sql('DELETE FROM governance_prod.metrics.ingestions')